# 99Acres Bengaluru Real Estate Analytics

## 1. Business Objective

The objective of this project is to analyze Bengaluru residential
property listings and identify patterns in:

- Property prices
- Price per square foot
- Locality
- Property size
- BHK configuration
- Property type
- Availability and listing concentration

### Key Business Questions

1. What does the Bengaluru residential property market look like?
2. Which localities have the highest number of listings?
3. Which localities have the highest property prices?
4. Which localities have the highest price per square foot?
5. How does property size influence price?
6. How does BHK configuration influence price?
7. Which locations appear relatively affordable?
8. Are there unusual or potentially anomalous property listings?

### Analytical Approach

The analysis will follow these stages:

1. Data acquisition
2. Data discovery
3. Data quality assessment
4. Data cleaning
5. Feature engineering
6. Exploratory data analysis
7. Market analysis
8. Business insights
9. Dashboard preparation

In [3]:
%pip install --upgrade jupyter ipywidgets tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import kagglehub

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

## 2. Data Source

The dataset used for this analysis is the Bengaluru 99Acres
property dataset available through Kaggle.

Dataset:
`rohan2662/99acres-bengaluru-dataset`

The dataset contains both cleaned and uncleaned versions of
the property listing data.

The uncleaned dataset will be used to understand data-quality
issues, while the cleaned dataset will be evaluated as a
reference for comparison.

In [4]:
DATASET = "rohan2662/99acres-bengaluru-dataset"

dataset_path = kagglehub.dataset_download(DATASET)

print("Dataset location:")
print(dataset_path)

Dataset location:
/Users/triguna/.cache/kagglehub/datasets/rohan2662/99acres-bengaluru-dataset/versions/1


In [5]:
files = os.listdir(dataset_path)

dataset_files = pd.DataFrame({
    "file_name": files
})

dataset_files

,file_name
0,bengaluru-properties-99acres.xlsx
1,bengaluru-properties-99acres(Uncleaned_data) (...


In [6]:
excel_files = [
    file for file in files
    if file.lower().endswith((".xlsx", ".xls"))
]

print(f"Excel files found: {len(excel_files)}")

for file in excel_files:
    print(f"• {file}")

Excel files found: 2
• bengaluru-properties-99acres.xlsx
• bengaluru-properties-99acres(Uncleaned_data) (1).xlsx


In [7]:
clean_file = os.path.join(
    dataset_path,
    "bengaluru-properties-99acres.xlsx"
)

raw_file = os.path.join(
    dataset_path,
    "bengaluru-properties-99acres(Uncleaned_data) (1).xlsx"
)

In [8]:
df = pd.read_excel(clean_file)

df_raw = pd.read_excel(raw_file)

## 3. Dataset Overview

Before performing any transformation, we first establish the
size, structure, and characteristics of the available datasets.

In [9]:
overview = pd.DataFrame({
    "Dataset": ["Raw / Uncleaned", "Cleaned"],
    "Rows": [df_raw.shape[0], df.shape[0]],
    "Columns": [df_raw.shape[1], df.shape[1]],
})

overview

,Dataset,Rows,Columns
0,Raw / Uncleaned,412,5
1,Cleaned,398,6


In [13]:
# Checking the columns in the cleaned dataset
columns = pd.DataFrame({
    "Column_Number": range(1, len(df.columns) + 1),
    "Column_Name": df.columns
})
columns

,Column_Number,Column_Name
0,1,name
1,2,location
2,3,price_lakhs
3,4,area_sqft
4,5,bhk
5,6,is_starred


In [12]:
print("Raw dataset columns:")
print(df_raw.columns.tolist())

Raw dataset columns:
['name', 'location', 'price', 'area', 'bhk']


In [14]:
raw_columns = set(df_raw.columns)
clean_columns = set(df.columns)

print("Columns only in RAW:")
print(raw_columns - clean_columns)

print("\nColumns only in CLEANED:")
print(clean_columns - raw_columns)

Columns only in RAW:
{'price', 'area'}

Columns only in CLEANED:
{'price_lakhs', 'area_sqft', 'is_starred'}


In [15]:
data_types = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notna().sum().values,
    "Null_Count": df.isna().sum().values
})

display(data_types)

,Column,Data_Type,Non_Null_Count,Null_Count
0,name,object,398,0
1,location,object,398,0
2,price_lakhs,float64,398,0
3,area_sqft,int64,398,0
4,bhk,int64,398,0
5,is_starred,int64,398,0


In [16]:
display(df.head())

,name,location,price_lakhs,area_sqft,bhk,is_starred
0,sobha royal pavilion,sarjapur road,240.0,1507,3,1
1,sobha royal pavilion,"carmelaram, sarjapur road",288.0,1800,3,1
2,sobha windsor,whitefield,210.0,1550,3,1
3,"vajarahalli, bangalore, bangalore south",vajarahalli,445.0,1500,4,0
4,sobha royal pavilion,sarjapur road,380.0,2232,4,1


In [17]:
display(df.sample(5))

,name,location,price_lakhs,area_sqft,bhk,is_starred
124,pride pegasus,hennur gardens,195.0,1629,3,0
207,hiranandani glen gate,hebbal,270.0,1665,3,1
396,samruddhi,uttarahalli,95.0,1320,3,0
221,provident sunworth,mysore road,57.8,1100,3,1
273,prestige jindal city,tumkur road,126.0,1073,2,1


In [18]:
display(df.head().T)

,0,1,2,3,4
name,sobha royal pavilion,sobha royal pavilion,sobha windsor,"vajarahalli, bangalore, bangalore south",sobha royal pavilion
location,sarjapur road,"carmelaram, sarjapur road",whitefield,vajarahalli,sarjapur road
price_lakhs,240.0,288.0,210.0,445.0,380.0
area_sqft,1507,1800,1550,1500,2232
bhk,3,3,3,4,4
is_starred,1,1,1,0,1


RAW COLUMNS
name
location
price
area
bhk

CLEANED COLUMNS
name
location
price_lakhs
area_sqft
bhk
is_starred


In [20]:
print("RAW COLUMNS")
print("=" * 50)

for column in df_raw.columns:
    print(column)

print("\nCLEANED COLUMNS")
print("=" * 50)

for column in df.columns:
    print(column)

RAW COLUMNS
name
location
price
area
bhk

CLEANED COLUMNS
name
location
price_lakhs
area_sqft
bhk
is_starred


In [21]:
print("\nRAW DATA")
display(df_raw.head().T)

print("\nCLEANED DATA")
display(df.head().T)


RAW DATA


,0,1,2,3,4
name,Sobha Royal Pavilion\n4.2,Sobha Royal Pavilion\n4.2,Sobha Windsor\n4.3,"Vajarahalli, Bangalore, Bangalore South",Sobha Royal Pavilion\n4.2
location,"3 BHK Flat in Sarjapur Road, Bangalore","3 BHK Flat in Carmelaram, Sarjapur Road","3 BHK Flat in Whitefield, Bangalore","4 Bedroom House in Vajarahalli, Bangalore","4 BHK Flat in Sarjapur Road, Bangalore"
price,₹2.4 Cr,₹2.88 Cr,₹2.1 Cr,₹4.45 Cr,₹3.8 Cr
area,"1,507 sqft","1,800 sqft","1,550 sqft","1,500 sqft","2,232 sqft"
bhk,3 BHK,3 BHK,3 BHK,4 BHK,4 BHK



CLEANED DATA


,0,1,2,3,4
name,sobha royal pavilion,sobha royal pavilion,sobha windsor,"vajarahalli, bangalore, bangalore south",sobha royal pavilion
location,sarjapur road,"carmelaram, sarjapur road",whitefield,vajarahalli,sarjapur road
price_lakhs,240.0,288.0,210.0,445.0,380.0
area_sqft,1507,1800,1550,1500,2232
bhk,3,3,3,4,4
is_starred,1,1,1,0,1
